# Convert data from InSituPy into SpatialData

## Setup and Imports

In [1]:
# Enable autoreload for development
%load_ext autoreload
%autoreload 2

## Make sure `SpatialData` is installed

If it is not installed yet, install it with:
```bash
pip install spatialdata[extra]
```
InSituPy pins `spatialdata>=0.8.0,<0.9.0` (see `pyproject.toml`) - the export is developed and tested against `spatialdata==0.8.0`, paired with a pinned `zarr>=3.2.1,<4.0.0` so the exported store is deterministically zarr v3. For more information on the installation of `SpatialData` see [here](https://spatialdata.scverse.org/en/stable/installation.html).

In [2]:
from pathlib import Path

from insitupy import CACHE, InSituData, InSituExperiment
from insitupy.spatialdata import convert_to_spatialdata

## Load InSituPy Data

First, let's load some example data. We'll demonstrate conversion with both:
- `InSituData`: A single spatial sample
- `InSituExperiment`: A collection of multiple samples

### Loading a Single Sample (InSituData)

In [3]:
# Load a single InSituData object
data_dir = Path(CACHE / "out/demo_insitupy_project")
xd = InSituData.read(data_dir)
xd.load_all()

In [4]:
# Display the InSituData object
xd

InSituData
Method:		Xenium
Slide ID:	0001879
Sample ID:	Replicate 1
UID:		None
Path:		C:\Users\ge37voy\.cache\InSituPy\out\demo_insitupy_project

    ➤ images
       'CD20':     (25778, 35416)
       'HE':       (25778, 35416, 3)
       'HER2':     (25778, 35416)
       'nuclei':   (25778, 35416)
    ➤ cells
       MultiCellData with main layer 'main'
           table
               AnnData object with n_obs × n_vars = 157600 × 297
               obs: 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'n_genes_by_counts', 'n_genes', 'leiden'
               var: 'gene_ids', 'feature_types', 'genome', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells'
               uns: 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'pca', 'umap'
               obsm: 'X_pca', 'X_umap', 'spatial'
               varm: 'PCs'
               layers: 'counts', 'norm_counts'
               obsp: 'connectivit

### Create dataset with multiple samples (InSituExperiment)

**Note on Transcripts**: Historically, a `dask` version conflict between `spatialdata` and `dask-geopandas` made transcript cropping with dask DataFrames very inefficient whenever `spatialdata` was also installed. As of `spatialdata==0.8.0` (the version InSituPy now requires) this dependency conflict is resolved - both packages install and import together cleanly. The cropping performance itself has not been re-benchmarked here, so we still delete the transcripts beforehand to keep this demo fast and sidestep the workflow entirely for `from_regions()`.

To avoid performance issues when creating an `InSituExperiment` from regions (which involves cropping), we delete the transcripts beforehand.

In [ ]:
# del xd.transcripts

In [5]:
exp = InSituExperiment.from_regions(
    data=xd,
    region_key="TMA"  # Column in annotations containing region IDs
)

C:\Users\ge37voy\AppData\Local\Temp\ipykernel_15436\817893223.py:1: UserWarning: Transcript data will be loaded into memory to speed up region cropping. This may require substantial RAM for large datasets.
  exp = InSituExperiment.from_regions(


2026-07-11 21:59:40 | [INFO] Loading transcripts into memory...
2026-07-11 21:59:43 | [INFO] Transcripts loaded: 42,638,083 rows.


Iterating regions: 100%|██████████| 6/6 [00:19<00:00,  3.27s/it]


In [6]:
# Display the experiment
exp

InSituExperiment (insitupy mode)
Path:		None
    ➤ data
        6 samples
        3 metadata columns:
        "uid", "region_key", "region_name"
        Loaded modalities
            cells: 6/6
            images: 6/6
            transcripts: 6/6
            annotations: 3/6
            regions: 6/6
    ➤ filters
        Base filters: none
        
        Composite filters: none
    ➤ table
        no tables built

## Convert to SpatialData

The `convert_to_spatialdata()` function handles the conversion of all data modalities into SpatialData elements. The full, versioned specification of this naming dialect lives in `insitupy/spatialdata/DIALECT.md`; a summary follows.

### Element naming convention

```
{SAMPLE.<uid>..}?<MODALITY>.<locator>[.<locator>...]
```

The `SAMPLE.<uid>..` prefix (note the trailing double dot) is added only when converting an `InSituExperiment`; a single `InSituData` produces un-prefixed keys.

**Single Sample (`InSituData`):**

- `IMAGES.CD20` - CD20 staining image
- `CELLS.main.table` - main cell expression table
- `CELLS.main.circles` / `CELLS.main.circles_sized` - cell centroid circles
- `CELLS.main.boundaries.cells` - cell boundary masks
- `UNITS.<key>.table` / `UNITS.<key>.shapes` - spatial units (e.g. Visium spots), if present - see §3.3 below
- `TRANSCRIPTS` - transcript point data (omit entirely with `include_transcripts=False`, see §3.4)
- `ANNOTATIONS.Demo` / `REGIONS.TMA` - annotation / region shapes

**Multiple Samples (`InSituExperiment`):**

Every key above is prefixed with `SAMPLE.<uid>..`, e.g. `SAMPLE.1f1bcf1f..IMAGES.CD20`, `SAMPLE.1f1bcf1f..CELLS.main.table`.

The store also carries a versioned dialect descriptor at `sdata.attrs["insitupy_spatialdata_dialect"]`, so a reader can detect "this store is InSituPy dialect, version N" without parsing element-name strings - demonstrated in §3.4.

### 3.1 Convert individual sample

In [7]:
# Convert InSituData to SpatialData
sdata = convert_to_spatialdata(xd)

2026-07-11 22:00:48 | [INFO] No case-insensitive conflicts found.


In [8]:
# Display the SpatialData object
sdata

SpatialData object
├── Images
│     ├── 'IMAGES.CD20': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     ├── 'IMAGES.HE': DataTree[cyx] (3, 25778, 35416), (3, 12889, 17708), (3, 6444, 8854), (3, 3222, 4427), (3, 1611, 2213), (3, 805, 1106)
│     ├── 'IMAGES.HER2': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     └── 'IMAGES.nuclei': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
├── Labels
│     ├── 'CELLS.main.boundaries.cells': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
│     └── 'CELLS.main.boundaries.nuclei': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
├── Points
│     └── 'TRANSCRIPTS': DataFrame with shape: (42638083, 8) (3D points)
├── Shapes
│     ├── '

### 3.2 Convert Experiment (Multiple Samples)

In [9]:
# Convert InSituExperiment to SpatialData
sdexp = convert_to_spatialdata(exp)

2026-07-11 22:01:21 | [INFO] No case-insensitive conflicts found.


In [10]:
# Display the experiment SpatialData
sdexp

SpatialData object
├── Images
│     ├── 'SAMPLE.4bd78871..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.4bd78871..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.4bd78871..IMAGES.HER2': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.4bd78871..IMAGES.nuclei': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.8bd49b5f..IMAGES.CD20': DataTree[cyx] (1, 4706, 4706), (1, 2353, 2353), (1, 1176, 1176), (1, 588, 588), (1, 294, 294), (1, 147, 147)
│     ├── 'SAMPLE.8bd49b5f..IMAGES.HE': DataTree[cyx] (3, 4706, 4706), (3, 2353, 2353), (3, 1176, 1176), (3, 588, 588), (3, 294, 294), (3, 147, 147)
│     ├── 'SAMPLE.8bd49b5f..IMAGES.HER2': DataTree[cyx] (1, 4706, 

### 3.3 Spatial units export

Spatial units (e.g. Visium spots, or any other polygon-based unit layer added via `InSituData.add_units()`) are exported as a `TableModel` + `ShapesModel` pair, following the same `region`/`region_key`/`instance_key` linkage as cells - but using the units' own real polygon geometries directly instead of synthesizing circles from centroids.

The demo project loaded above has no units attached, so this section loads a small, self-contained Visium dataset instead (images are dropped here purely to keep the cell fast and focused on the `units` modality).

In [11]:
from insitupy.datasets import visium_human_breast_cancer

visium = visium_human_breast_cancer()
del visium.images  # keep this demo focused on the units modality

sdata_units = convert_to_spatialdata(visium)
sdata_units

2026-07-11 22:01:38 | [INFO] H5 file exists. Checking md5sum...
2026-07-11 22:01:38 | [INFO] The h5 file md5sum matches. Download is skipped. To force download set `overwrite=True`.
2026-07-11 22:01:38 | [INFO] Spatial directory exists. Download is skipped. To force download set `overwrite=True`.
2026-07-11 22:01:38 | [INFO] Visium data structure is ready at C:\Users\ge37voy\.cache\InSituPy\demo_datasets\visium_hbreastcancer\CytAssist_FFPE_Human_Breast_Cancer
2026-07-11 22:01:38 | [INFO] Dataset contains:
2026-07-11 22:01:38 | [INFO] - filtered_feature_bc_matrix.h5
2026-07-11 22:01:38 | [INFO] - spatial/ directory
2026-07-11 22:01:38 | [INFO] Reading Visium data with spatialdata-io...


c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\site-packages\spatialdata\models\models.py:1267: UserWarning: Converting `region_key: region` to categorical dtype.
  convert_region_column_to_categorical(adata)


2026-07-11 22:01:39 | [INFO] Using 'visium' coordinate system for pixel size extraction.
2026-07-11 22:01:39 | [INFO] Adding images...
2026-07-11 22:01:40 | [INFO] Converting 4992 Point geometries with radius to circular polygons using buffer.
2026-07-11 22:01:40 | [WARNING] Indices in `.shapes` do not match `.data.obs_names`. Shapes will be renamed according to the `obs_names`. For this to be valid, please make sure that the order of elements in `.shapes` and `.data` matches.
2026-07-11 22:01:40 | [INFO] Cleared modality 'images'.
2026-07-11 22:01:40 | [INFO] No case-insensitive conflicts found.


SpatialData object
├── Shapes
│     └── 'UNITS.visium.shapes': GeoDataFrame shape: (4992, 2) (2D shapes)
└── Tables
      └── 'UNITS.visium.table': AnnData (4992, 18085)
with coordinate systems:
    ▸ 'visium', with elements:
        UNITS.visium.shapes (Shapes)
    ▸ 'visium_downscaled_hires', with elements:
        UNITS.visium.shapes (Shapes)
    ▸ 'visium_downscaled_lowres', with elements:
        UNITS.visium.shapes (Shapes)

### 3.4 Skipping transcripts and reading the dialect descriptor

Transcript export (`PointsModel.parse(..., sort=True)`) is the most expensive step for large experiments. Pass `include_transcripts=False` to skip it entirely - useful when you only need cells, images, and/or shapes, or when transcript export is too slow for your use case.

Every export also carries a versioned dialect descriptor at `sdata.attrs["insitupy_spatialdata_dialect"]`, letting a reader detect "this store is InSituPy dialect, version N" without parsing element-name strings.

In [12]:
from insitupy.datasets import xenium_test_dataset_v3_mm

xd_small = xenium_test_dataset_v3_mm()

sdata_with_tx = convert_to_spatialdata(xd_small)
sdata_without_tx = convert_to_spatialdata(xd_small, include_transcripts=False)

print(f"With transcripts (default):  {len(sdata_with_tx.points)} points element(s)")
print(f"include_transcripts=False:   {len(sdata_without_tx.points)} points element(s)")
print()
print("Dialect descriptor:", sdata_with_tx.attrs["insitupy_spatialdata_dialect"])

2026-07-11 22:04:52 | [INFO] This dataset exists already. Download is skipped. To force download set `overwrite=True`.
2026-07-11 22:04:52 | [INFO] Reading Xenium data with InSituPy backend...
2026-07-11 22:04:52 | [INFO] Loading cells...
2026-07-11 22:04:52 | [INFO] Loading images...
2026-07-11 22:04:52 | [WARNING] 'morphology_focus_0000.ome.tif' is part of a multi-file OME-TIFF. Axes are inferred from this file only and only data from this file will be returned.
2026-07-11 22:04:52 | [INFO] Loading transcripts...
2026-07-11 22:04:52 | [INFO] For this dataset no additional images are available.
2026-07-11 22:04:52 | [INFO] No case-insensitive conflicts found.
2026-07-11 22:04:52 | [INFO] No case-insensitive conflicts found.
With transcripts (default):  1 points element(s)
include_transcripts=False:   0 points element(s)

Dialect descriptor: {'version': 1, 'modalities': ['cells', 'units', 'images', 'transcripts', 'annotations', 'regions'], 'sample_prefix_pattern': 'SAMPLE.<uid>..'}


## Saving and Loading SpatialData

SpatialData objects can be saved to disk in Zarr format for efficient storage and lazy loading.

In [13]:
# Define output paths
outpath = CACHE / "test_spatialdata.zarr"
exp_outpath = CACHE / "exp_spatialdata.zarr"

In [14]:
# Save single sample SpatialData
sdata.write(outpath, overwrite=True)
print(f"Saved to: {outpath}")

c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: 

Saved to: C:\Users\ge37voy\.cache\InSituPy\test_spatialdata.zarr


In [15]:
# Save experiment SpatialData
sdexp.write(exp_outpath, overwrite=True)
print(f"Saved to: {exp_outpath}")

c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: zarr v3 autosharding will be the default in the next minor release.
  return next(self.gen)
c:\Users\ge37voy\AppData\Local\miniconda3\envs\isp012\Lib\contextlib.py:141: UserWarning: 

Saved to: C:\Users\ge37voy\.cache\InSituPy\exp_spatialdata.zarr


### Load from Disk

In [16]:
from spatialdata import SpatialData

# Load saved SpatialData
sdata_loaded = SpatialData.read(outpath)
sdata_loaded

2026-07-11 22:13:36 | [INFO] root_attr: version
2026-07-11 22:13:36 | [INFO] root_attr: multiscales
2026-07-11 22:13:36 | [INFO] root_attr: omero
2026-07-11 22:13:36 | [INFO] datasets [{'path': 's0', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 1.0, 1.0]}, {'type': 'translation', 'translation': [0.0, 0.0, 0.0]}]}, {'path': 's1', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 2.0, 2.0]}, {'type': 'translation', 'translation': [0.0, 0.5, 0.5]}]}, {'path': 's2', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 4.000310366232154, 4.0]}, {'type': 'translation', 'translation': [0.0, 1.5001551831160769, 1.5]}]}, {'path': 's3', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 8.000620732464307, 8.0]}, {'type': 'translation', 'translation': [0.0, 3.5003103662321537, 3.5]}]}, {'path': 's4', 'coordinateTransformations': [{'type': 'scale', 'scale': [1.0, 16.001241464928615, 16.003615002259377]}, {'type': 'translation', 'translation': [0

SpatialData object, with associated Zarr store: C:\Users\ge37voy\.cache\InSituPy\test_spatialdata.zarr
├── Images
│     ├── 'IMAGES.CD20': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     ├── 'IMAGES.HE': DataTree[cyx] (3, 25778, 35416), (3, 12889, 17708), (3, 6444, 8854), (3, 3222, 4427), (3, 1611, 2213), (3, 805, 1106)
│     ├── 'IMAGES.HER2': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
│     └── 'IMAGES.nuclei': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213), (1, 805, 1106)
├── Labels
│     ├── 'CELLS.main.boundaries.cells': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
│     └── 'CELLS.main.boundaries.nuclei': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213), (805, 1106)
├── Points
│     └── '